In [17]:
import os

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [19]:
%load_ext dotenv
%dotenv
openai_api_key = os.getenv("api_key")

In [20]:
chat_template_books = ChatPromptTemplate.from_template(''' 
Suggest three of the best intermediate-level {programming language} books.
Answer only by listing the books.
'''
)

chat_template_project = ChatPromptTemplate.from_template(''' 
Suggest three interesting {programming language} projects suitable for intermediate-level programmers.
Answer only by listing the projects.
'''
)

chat_template_time = ChatPromptTemplate.from_template('''
I'm an intermediate level programmer.

Consider the following literature:
{books}

Also consider the following projects:
{projects}

Roughly how much time would it take me to complete the literature and the projects?
'''
)

In [21]:
chat = ChatOpenAI(model_name="gpt-4", 
                  seed=365,
                  temperature=0,
                  max_tokens=500,
                api_key=openai_api_key)

In [22]:
string_parser = StrOutputParser()

In [23]:
chain_books = (chat_template_books | chat | string_parser)
chain_projects = (chat_template_project | chat | string_parser)


In [24]:
chain_parallel = RunnableParallel({"books": chain_books, "projects": chain_projects})

In [25]:
chain_parallel.invoke({"programming language": "Python"})

{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Weather Forecasting Application using APIs\n2. Developing a Web Scraper for E-commerce Websites\n3. Creating a Personal Finance Tracker with Data Visualization Features'}

In [32]:
chain_time1 = (RunnableParallel({"books": chain_books, "projects": chain_projects}) | (chat_template_time | chat | string_parser))

In [36]:
chain_time2 = ({"books": chain_books, "projects": chain_projects}) | (chat_template_time | chat | string_parser)

In [37]:
print(chain_time2.invoke({"programming language": "Python"}))

The time it takes to complete the literature and projects can vary greatly depending on several factors such as your reading speed, comprehension level, familiarity with the subject matter, and the amount of time you can dedicate each day. However, here's a rough estimate:

1. "Fluent Python: Clear, Concise, and Effective Programming" - This book is around 800 pages. If you read and practice for about 2 hours a day, it might take you around 1.5 to 2 months.

2. "Python Cookbook: Recipes for Mastering Python 3" - This book is around 700 pages. Following the same reading and practicing schedule, it might take you around 1 to 1.5 months.

3. "Effective Python: 90 Specific Ways to Write Better Python" - This book is around 230 pages. It might take you around 2 to 3 weeks.

For the projects:

1. Building a Weather Forecasting Application using APIs - Depending on the complexity and features, it might take you around 1 to 2 weeks.

2. Developing a Web Scraper for E-commerce Websites - This c

In [35]:

chain_time2.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            